Este Notebook tiene como objetivo aportar interpretabilidad a los modelos de los grafos.

Se analiza qué factores explican la aparición de una arista, desde el punto de vista TF-IDF (léxico) como SBERT (semántico).

In [34]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
# Ruta del proyecto en Google Drive
PROJECT_DIR = "/content/drive/MyDrive/TFG-FakeNewsNet"

# Me muevo a la carpeta principal del proyecto
%cd $PROJECT_DIR

!ls

/content/drive/MyDrive/TFG-FakeNewsNet
data  models  notebooks  README.md  README.md.gdoc  requirements.txt  results


In [36]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import load_npz
import joblib

In [37]:
edges_tfidf = pd.read_csv("results/content_edges_tfidf.csv")   # Cargo las aristas generadas en el Notebook 03 (TF-IDF)
edges_sbert = pd.read_csv("results/content_edges_sbert.csv")   # Cargo las aristas generadas en el Notebook 04 (SBERT)

tfidf_matrix = load_npz("data/tfidf_matrix.npz")   # Cargo la matriz TF-IDF

df = pd.read_csv("data/noticias_preproc.csv")   # Cargo el dataset preprocesado generado en el Notebook 02
df["domain"] = df["subject"]

tfidf_vectorizer = joblib.load("models/tfidf_vectorizer.joblib")   # Cargo el vector TF-IDF generado en el Notebook 03


# Reconstrucción subject

Reconstruyo los dominios a partir de las aristas del grafo TF-IDF. Esto garantiza que el análisis dependa de los resultados finales.

In [40]:
subjects = sorted(
    set(edges_tfidf["domain_A"]).union(edges_tfidf["domain_B"])   # Saco los dominios que hay en las aristas del grafo
)
subjects


['Government News',
 'Middle-east',
 'News',
 'US_News',
 'left-news',
 'politics',
 'politicsNews',
 'worldnews']

# Explainability del grafo TF-IDF

Identifica los términos léxicos que explican la similitud entre 2 dominios, a partir del vocabulario compartido en sus noticias.

In [44]:
def explain_tfidf_edge(domain_a, domain_b, df, tfidf_matrix, vectorizer, top_k=10):

    idx_a = df[df["domain"] == domain_a].index   # Selecciono las noticias pertenecientes al dominio (en este caso el A)
    idx_b = df[df["domain"] == domain_b].index

    v_a = tfidf_matrix[idx_a].mean(axis=0)   # Extraigo las filas TF-IDF de cada dominio y calculo la media por columnas (para producir un vector)
    v_b = tfidf_matrix[idx_b].mean(axis=0)

    shared = np.multiply(v_a, v_b)  # Multiplico elementos de A y B para saber cuales son importantes

    feature_names = vectorizer.get_feature_names_out()   # Identifico la lista de términos del vocabulario TF-IDF
    shared = np.asarray(shared).ravel()

    top_idx = np.argsort(shared)[-top_k:][::-1]   # Me quedo con los términos dominantes y en orden descendente

    return [(feature_names[i], shared[i]) for i in top_idx if shared[i] > 0]


Ejemplo:

In [45]:
domain_a = "US_News"
domain_b = "politicsNews"

explain_tfidf_edge(
    domain_a,
    domain_b,
    df,
    tfidf_matrix,
    tfidf_vectorizer,
    top_k=10
)


[('trump', np.float64(0.0019066366593340216)),
 ('new', np.float64(0.0006085548343824666)),
 ('clinton', np.float64(0.0003885296356784015)),
 ('president', np.float64(0.0003409051636736076)),
 ('washington', np.float64(0.0002700572293916969)),
 ('election', np.float64(0.00025222945183000985)),
 ('state', np.float64(0.0002487493476357339)),
 ('russia', np.float64(0.00022861168827011658)),
 ('obama', np.float64(0.00018201521027716962)),
 ('say', np.float64(0.00017508523080566804))]

# Explainability del grafo SBERT

Identifica ejemplos de noticias de dominios distintos con alto grado de similitud semántica.

In [46]:
embeddings = np.load("data/embeddings_sbert.npy")
df["embed"] = list(embeddings)   # Asocio cada embedding a su noticia correspondiente


In [53]:
def explain_sbert_edge(domain_a, domain_b, df, top_k=2):


    emb_a = np.vstack(df[df["domain"] == domain_a]["embed"].values)  # Extrae los embeddings por dominio (en este caso el A)
    emb_b = np.vstack(df[df["domain"] == domain_b]["embed"].values)

    sim = cosine_similarity(emb_a, emb_b)

    idx = np.unravel_index(np.argsort(sim.ravel())[-top_k:], sim.shape)  # Saca los pares de noticias más parecidos semánticamente

    examples = []
    for i, j in zip(idx[0], idx[1]):
        text_a = df[df["domain"] == domain_a].iloc[i]["text"]  # Recupera el texto original de la noticia
        text_b = df[df["domain"] == domain_b].iloc[j]["text"]
        examples.append((sim[i, j], text_a[:300], text_b[:300]))  # Guarda la similitud y ambos textos (quito 300 caracteres para no saturar el Notebook)

    return examples



Ejemplo:

In [54]:
examples = explain_sbert_edge("Middle-east", "US_News", df, top_k=2)

for sim, a, b in examples:
    print(f"\nSimilitud: {sim:.3f}")
    print("A:", a)
    print("B:", b)




Similitud: 1.000
A: Shawn Helton 21st Century WireTruth is often stranger than fiction when looking at the bizarre phenomena surrounding many mass casualty incidents   and the Orlando Pulse nightclub shooting was no exception.It was recently revealed that the world s largest security firm G4S, who had employed the man 
B: Shawn Helton 21st Century WireTruth is often stranger than fiction when looking at the bizarre phenomena surrounding many mass casualty incidents   and the Orlando Pulse nightclub shooting was no exception.It was recently revealed that the world s largest security firm G4S, who had employed the man 

Similitud: 1.000
A: Patrick Henningsen and Shawn Helton 21st Century WireOnce again, we ve arrived at our New Years Eve wrap-up of some of the most compelling and conspiratorial stories of the year. Like in years past, 2017 presented a polarizing political landscape, further exposing the current establishment paradigm.
B: Patrick Henningsen and Shawn Helton 21st Century 